In [1]:
DATA = '../data/'
FIGS = '../results/figures/'
CACHE = '../f1_cache'

In [2]:
import fastf1, pandas, sklearn, matplotlib

/Users/huytran/Library/Python/3.9/lib/python/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


In [3]:
import urllib3
r = urllib3.PoolManager().request('GET', 'https://api.jolpi.ca/ergast/f1/2025.json')
print(r.status)

200


In [4]:
import fastf1, numpy as np, os
# fastf1 - F1 Timing Data Library
# numpy (NumPy) - Library for Multidimensional Array & Matrices + The mathematical functions to operate them

os.makedirs(CACHE, exist_ok=True) # make new folder named f1_cache if doesn't exist. CACHE comes from cell 1
fastf1.Cache.enable_cache(CACHE) # f1_cache is the storage place for data so they can be found and used quickly again later

s = fastf1.get_session(2025, 'Barcelona', 'R') # SELECT the race session, in this case Zandvoort 2025. Nothing is downloaded yet
s.load(telemetry=False, messages=False) # this line is what downloads. telemetry = per-lap sensor traces (speed, throttle, brake). messages = the race control text feed (flags, penalties, investigations). Neither is needed, we only want one row per lap
laps = s.laps # a DataFrame - one row per driver per lap

# print relatable session data to manually determine if race session suitable
print("rainfall:", s.weather_data['Rainfall'].sum())
print(laps['TrackStatus'].value_counts())
print("green %:", round((laps['TrackStatus'] == '1').mean(), 2))
print(laps.groupby('Driver')['Stint'].max().value_counts())

# keep only laps that are BOTH trustworthy and raced under normal conditions. Two separate filters:
#   IsAccurate      - True when the lap has a real recorded time, is not an in-lap or out-lap, and has no pit time attached
#   TrackStatus '1' - the lap ran entirely under green flag
clean = laps[laps['IsAccurate'] & (laps['TrackStatus'] == '1')].copy()
clean['Sec'] = clean['LapTime'].dt.total_seconds()

# THE DEGRADATION GATE - the reason this whole step exists.
# For each tyre compound, draw a straight line through (tyre age, lap time) and read its slope.
# The slope is seconds lost per lap of tyre age. If it is not positive, tyres are not visibly wearing
# in this race, and the app has nothing to simulate. Switch races.
for comp in clean['Compound'].dropna().unique():  # every compound used in the race. dropna() skips laps with no compound recorded
    sub = clean[clean['Compound'] == comp]        # narrow to just the laps run on that one compound
    if len(sub) >= 30:                            # fewer than 30 laps and the line fit is too noisy to believe
        slope = np.polyfit(sub['TyreLife'], sub['Sec'], 1)[0]  # fit a degree-1 line. Returns [slope, intercept], so [0] is the slope
        print(f"{comp}: {slope:+.3f} s/lap  (n={len(sub)})")   # :+ forces the sign to show, .3f is 3 decimal places, n is how many laps backed the number

core           INFO 	Loading data for Spanish Grand Prix - Race [v3.7.0]
req            INFO 	Using cached data for session_info
req            INFO 	Using cached data for driver_info
req            INFO 	Using cached data for session_status_data
req            INFO 	Using cached data for lap_count
req            INFO 	Using cached data for track_status_data
req            INFO 	Using cached data for _extended_timing_data
req            INFO 	Using cached data for timing_app_data
core           INFO 	Processing timing data...
req            INFO 	Using cached data for weather_data
core           INFO 	Finished loading data for 19 drivers: ['81', '4', '16', '63', '27', '44', '6', '10', '14', '1', '30', '5', '22', '55', '43', '31', '87', '12', '23']


rainfall: 0
TrackStatus
1      1088
4        79
124      17
41       17
12        1
24        1
Name: count, dtype: int64
green %: 0.9
Stint
4.0    12
3.0     5
5.0     2
Name: count, dtype: int64
SOFT: +0.058 s/lap  (n=526)
MEDIUM: +0.022 s/lap  (n=457)


In [5]:
from sklearn.linear_model import LinearRegression

for comp in clean['Compound'].dropna().unique():
    sub = clean[clean['Compound'] == comp].dropna(subset=['TyreLife', 'LapNumber', 'Sec'])
    if len(sub) >= 30:
        m = LinearRegression().fit(sub[['TyreLife', 'LapNumber']], sub['Sec'])
        print(f"{comp}: tyre {m.coef_[0]:+.3f}  fuel {m.coef_[1]:+.3f}  (n={len(sub)})")

SOFT: tyre +0.093  fuel -0.057  (n=526)
MEDIUM: tyre +0.063  fuel -0.053  (n=457)
